# 03 — Anomaly Detection
Compares absolute Z-score flags with Isolation Forest results.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import zscore
from sklearn.ensemble import IsolationForest
root=Path.cwd(); clean=root/'data'/'clean'; pl=pd.read_csv(clean/'profitandloss.csv'); bs=pd.read_csv(clean/'balancesheet.csv')
frame=pl.merge(bs[['symbol','year_label','debt_to_equity']],on=['symbol','year_label'],how='left'); features=['sales','net_profit','opm_pct','debt_to_equity']; x=frame[features].replace([np.inf,-np.inf],np.nan).fillna(frame[features].median())
z=np.abs(zscore(x,nan_policy='omit')); frame['z_score_max']=np.nanmax(z,axis=1); frame['zscore_flag']=frame.z_score_max>=3
frame['isolation_flag']=IsolationForest(contamination=.03,random_state=42).fit_predict(x)==-1; frame['methods_agree']=frame.zscore_flag==frame.isolation_flag
out=root/'data'/'warehouse'/'anomaly_flags.csv'; frame.loc[frame.zscore_flag|frame.isolation_flag,['symbol','year_label','z_score_max','zscore_flag','isolation_flag','methods_agree']].to_csv(out,index=False); display(pd.read_csv(out).sort_values('z_score_max',ascending=False)); print(out)